In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import time
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import cudf
import cupy as cp
from cuml.preprocessing import TargetEncoder
from cuml.linear_model import LogisticRegression as cuLogReg

import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# ============================================================
# GLOBAL CONFIG
# ============================================================
TRAIN_PATH   = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH    = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
TARGET       = "Churn"
DROP_COLS    = ["customerID", "id"]
N_SPLITS     = 5
SEED         = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# LOAD
# ============================================================
print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"  train: {train_df.shape}  test: {test_df.shape}")

y = (train_df[TARGET].astype(str).str.strip().str.lower() == "yes").astype(np.int32).values
N = len(train_df)

# ============================================================
# FEATURE ENGINEERING
# ============================================================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["TotalCharges"]   = pd.to_numeric(df["TotalCharges"],   errors="coerce").fillna(0.0)
    df["tenure"]         = pd.to_numeric(df["tenure"],         errors="coerce").fillna(0.0)
    df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce").fillna(0.0)

    df["charges_per_tenure"]   = df["MonthlyCharges"] / (df["tenure"] + 1)
    df["total_vs_expected"]    = df["TotalCharges"] / (df["MonthlyCharges"] * df["tenure"] + 1)
    df["total_minus_expected"] = df["TotalCharges"] - (df["MonthlyCharges"] * df["tenure"])

    service_cols = [
        "PhoneService", "MultipleLines", "OnlineSecurity",
        "OnlineBackup", "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies",
    ]
    for c in service_cols:
        if c in df.columns:
            df[c + "_bin"] = (df[c].astype(str).str.strip().str.lower() == "yes").astype(np.int32)
    bin_cols = [c + "_bin" for c in service_cols if c in df.columns]
    df["n_services"]       = df[bin_cols].sum(axis=1)
    df["is_high_value"]    = ((df["MonthlyCharges"] > df["MonthlyCharges"].median()) & (df["tenure"] > 24)).astype(np.int32)
    df["is_new_customer"]  = (df["tenure"] <= 3).astype(np.int32)
    df["is_loyal_customer"]= (df["tenure"] > 36).astype(np.int32)
    df["is_monthly"]       = (df["Contract"].astype(str).str.strip().str.lower() == "month-to-month").astype(np.int32)
    df["monthly_charge_bin"] = pd.cut(df["MonthlyCharges"], bins=10, labels=False).fillna(0).astype(np.int32)
    df["tenure_bin"]         = pd.cut(df["tenure"],         bins=10, labels=False).fillna(0).astype(np.int32)
    df["is_autopay"]         = df["PaymentMethod"].astype(str).str.lower().str.contains("automatic").astype(np.int32)

     # ============================================================
      # CONTRACT / SERVICE INTERACTIONS
       # ============================================================

    # map contract to numeric length
    contract_map = {
    "month-to-month": 1,
    "one year": 12,
    "two year": 24
     }

    contract_clean = df["Contract"].astype(str).str.strip().str.lower()
    df["contract_length"] = contract_clean.map(contract_map).fillna(1).astype(np.int32)

# flag for monthly contracts
    df["monthly_contract_flag"] = (df["contract_length"] == 1).astype(np.int32)

# tenure relative to contract length
    df["tenure_contract_ratio"] = df["tenure"] / (df["contract_length"] + 1)

# spending intensity per service
    df["charges_per_service"] = df["MonthlyCharges"] / (df["n_services"] + 1)

# engagement: services × tenure
    df["services_tenure"] = df["n_services"] * df["tenure"]

# how much of expected contract completed
    df["contract_progress"] = df["tenure"] / (df["contract_length"] + 1)

# spending per contract cycle
    df["charges_per_contract"] = df["TotalCharges"] / (df["contract_length"] + 1)

    return df



train_fe = engineer_features(train_df)
test_fe  = engineer_features(test_df)

# ============================================================
# SHARED PREPROCESSING (label-encode for tree models)
# ============================================================
def preprocess_together(X_tr: pd.DataFrame, X_te: pd.DataFrame):
    X_tr = X_tr.copy()
    X_te = X_te.copy()

    drop = [c for c in DROP_COLS + [TARGET] if c in X_tr.columns]
    X_tr = X_tr.drop(columns=drop, errors="ignore")
    X_te = X_te.drop(columns=[c for c in DROP_COLS if c in X_te.columns], errors="ignore")

    n_tr  = len(X_tr)
    X_all = pd.concat([X_tr, X_te], axis=0, ignore_index=True)

    cat_cols = X_all.select_dtypes(include=["object", "category"]).columns.tolist()
    for c in cat_cols:
        s = X_all[c].astype(str).fillna("__MISSING__")
        cats = pd.Index(s.unique()).sort_values()
        X_all[c] = pd.Categorical(s, categories=cats).codes.astype(np.int32)

    for c in X_all.select_dtypes(include=[np.number]).columns:
        X_all[c] = pd.to_numeric(X_all[c], errors="coerce").fillna(0)

    return X_all.iloc[:n_tr].reset_index(drop=True), X_all.iloc[n_tr:].reset_index(drop=True)

X_train_tree, X_test_tree = preprocess_together(train_fe, test_fe)
print(f"Tree features: {X_train_tree.shape[1]}")


# ============================================================
# ============================================================
# MODEL 1: TE-PAIR LOGISTIC REGRESSION (your original)
# ============================================================
# ============================================================
print("\n" + "="*70)
print("MODEL 1: TE-PAIR LOGISTIC REGRESSION")
print("="*70)

TE_VER = 200
TE_N_FOLDS = 5

FEATURES_TEPAIR = [c for c in train_df.columns if c not in DROP_COLS + [TARGET]]

def label_encode_train_test(train_raw, test_raw, cols):
    train_out = train_raw.copy()
    test_out  = test_raw.copy()
    for c in cols:
        tr_s = train_out[c].astype("string").fillna("__MISSING__")
        te_s = test_out[c].astype("string").fillna("__MISSING__")
        all_vals = pd.concat([tr_s, te_s], axis=0)
        uniq = pd.Index(all_vals.unique())
        mapping = pd.Series(np.arange(len(uniq), dtype=np.int32), index=uniq)
        train_out[c] = tr_s.map(mapping).astype(np.int32)
        test_out[c]  = te_s.map(mapping).astype(np.int32)
    return train_out, test_out

train_enc_te, test_enc_te = label_encode_train_test(
    train_df[FEATURES_TEPAIR], test_df[FEATURES_TEPAIR], FEATURES_TEPAIR
)

n_feat_te = len(FEATURES_TEPAIR)
pair_cols_te = []
pair_names_te = []
for i in range(n_feat_te):
    for j in range(i + 1, n_feat_te):
        f1, f2 = FEATURES_TEPAIR[i], FEATURES_TEPAIR[j]
        pair_cols_te.append((f1, f2))
        pair_names_te.append(f"{f1}__x__{f2}")
n_pair_te = len(pair_cols_te)

X_all_g_te  = cudf.DataFrame({f: cudf.Series(train_enc_te[f].values).astype("int32") for f in FEATURES_TEPAIR})
X_test_g_te = cudf.DataFrame({f: cudf.Series(test_enc_te[f].values).astype("int32") for f in FEATURES_TEPAIR})

def _clip01(x, eps=1e-5): return np.clip(x, eps, 1.0 - eps)
def _logit(x, eps=1e-5):
    x = _clip01(x, eps)
    return np.log(x / (1.0 - x)).astype(np.float32)

def make_logit3_features(tr_m, va_m, te_m, eps=1e-5):
    z_tr = _logit(tr_m, eps=eps); z_va = _logit(va_m, eps=eps); z_te = _logit(te_m, eps=eps)
    return (np.hstack([z_tr, z_tr**2, z_tr**3]).astype(np.float32),
            np.hstack([z_va, z_va**2, z_va**3]).astype(np.float32),
            np.hstack([z_te, z_te**2, z_te**3]).astype(np.float32))

kf_te = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_tepair  = np.zeros(N, dtype=np.float32)
pred_tepair = np.zeros(len(test_df), dtype=np.float32)
fold_aucs_te = []

for fold, (tr_idx, va_idx) in enumerate(kf_te.split(np.zeros(N), y), 1):
    print(f"\n  [TE-Pair] Outer fold {fold}/{N_SPLITS}  tr={len(tr_idx)} va={len(va_idx)}")

    tr_idx_g = cudf.Series(tr_idx); va_idx_g = cudf.Series(va_idx)
    X_tr_g = X_all_g_te.take(tr_idx_g); X_va_g = X_all_g_te.take(va_idx_g)
    y_tr = y[tr_idx].astype(np.int32); y_va = y[va_idx].astype(np.int32)
    y_tr_g = cudf.Series(y_tr)

    tr_pair = np.zeros((len(tr_idx), n_pair_te), dtype=np.float32)
    va_pair = np.zeros((len(va_idx), n_pair_te), dtype=np.float32)
    te_pair = np.zeros((len(test_df),  n_pair_te), dtype=np.float32)

    te_enc = TargetEncoder(n_folds=TE_N_FOLDS, smooth=0, seed=SEED+fold,
                           split_method="random", stat="mean", output_type="cupy")

    for t, (f1, f2) in enumerate(pair_cols_te):
        if (t == 0) or ((t+1) % 200 == 0) or (t+1 == n_pair_te):
            print(f"    Pair {t+1:>6d}/{n_pair_te}: {f1} x {f2}")
        X_tr_ij = X_tr_g[[f1, f2]]; X_va_ij = X_va_g[[f1, f2]]; X_te_ij = X_test_g_te[[f1, f2]]
        tr_oof_cp = te_enc.fit_transform(X_tr_ij, y_tr_g)
        tr_pair[:, t] = cp.asnumpy(tr_oof_cp).ravel().astype(np.float32)
        te_enc.fit(X_tr_ij, y_tr_g)
        va_pair[:, t] = cp.asnumpy(te_enc.transform(X_va_ij)).ravel().astype(np.float32)
        te_pair[:, t] = cp.asnumpy(te_enc.transform(X_te_ij)).ravel().astype(np.float32)

    X_tr_raw, X_va_raw, X_te_raw = make_logit3_features(tr_pair, va_pair, te_pair)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr_raw).astype(np.float32)
    X_va_s = scaler.transform(X_va_raw).astype(np.float32)
    X_te_s = scaler.transform(X_te_raw).astype(np.float32)

    meta = cuLogReg(penalty="l2", C=0.5, max_iter=4000, tol=1e-4, fit_intercept=True, verbose=0)
    meta.fit(cp.asarray(X_tr_s), cp.asarray(y_tr))
    oof_va = cp.asnumpy(meta.predict_proba(cp.asarray(X_va_s))[:, 1]).astype(np.float32)
    oof_tepair[va_idx] = oof_va
    fold_auc = roc_auc_score(y_va, oof_va)
    fold_aucs_te.append(float(fold_auc))
    print(f"    [TE-Pair] Fold {fold} AUC: {fold_auc:.6f}")
    pred_tepair += cp.asnumpy(meta.predict_proba(cp.asarray(X_te_s))[:, 1]).astype(np.float32) / N_SPLITS

tepair_auc = roc_auc_score(y, oof_tepair)
print(f"\n  [TE-Pair] OOF AUC: {tepair_auc:.6f}  Folds: {[f'{a:.4f}' for a in fold_aucs_te]}")
np.save(f"oof_tepair_v{TE_VER}.npy",  oof_tepair)
np.save(f"pred_tepair_v{TE_VER}.npy", pred_tepair)


# ============================================================
# ============================================================
# MODEL 2: XGBOOST (your original, now with engineered features)
# ============================================================
# ============================================================
print("\n" + "="*70)
print("MODEL 2: XGBOOST")
print("="*70)

XGB_VER = 102

# Also prepare categorical-dtype version for XGBoost native cat support
def preprocess_xgb_categorical(X_tr: pd.DataFrame, X_te: pd.DataFrame):
    X_tr = X_tr.copy(); X_te = X_te.copy()
    drop = [c for c in DROP_COLS + [TARGET] if c in X_tr.columns]
    X_tr = X_tr.drop(columns=drop, errors="ignore")
    X_te = X_te.drop(columns=[c for c in DROP_COLS if c in X_te.columns], errors="ignore")
    n_tr = len(X_tr)
    X_all = pd.concat([X_tr, X_te], axis=0, ignore_index=True)

    if "TotalCharges" in X_all.columns:
        X_all["TotalCharges"] = pd.to_numeric(X_all["TotalCharges"], errors="coerce").fillna(0.0).astype("float32")

    cat_cols = set(X_all.select_dtypes(include=["object"]).columns.tolist())
    cat_cols |= set(X_all.select_dtypes(include=["category"]).columns.tolist())
    for c in sorted(cat_cols):
        s = X_all[c]
        if str(s.dtype) == "category": s = s.astype("object")
        non_null = s[~pd.isna(s)]
        cats = pd.Index(non_null.map(lambda v: str(v)).unique(), dtype="object").sort_values()
        dtype = pd.CategoricalDtype(categories=cats, ordered=False)
        X_all[c] = s.map(lambda v: np.nan if pd.isna(v) else str(v)).astype(dtype)

    X_tr2 = X_all.iloc[:n_tr].reset_index(drop=True)
    X_te2 = X_all.iloc[n_tr:].reset_index(drop=True)
    return X_tr2, X_te2

X_train_xgb, X_test_xgb = preprocess_xgb_categorical(train_fe, test_fe)

xgb_params = dict(
    n_estimators=100_000, learning_rate=0.1, max_depth=3,
    min_child_weight=5, subsample=0.85, colsample_bytree=0.85,
    objective="binary:logistic", eval_metric="auc",
    tree_method="hist", enable_categorical=True,
    random_state=SEED, n_jobs=-1, early_stopping_rounds=200, device="cuda",
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_xgb  = np.zeros(N, dtype=np.float32)
pred_xgb = np.zeros(len(test_df), dtype=np.float32)
fold_aucs_xgb = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_xgb, y), 1):
    X_tr, X_va = X_train_xgb.iloc[tr_idx], X_train_xgb.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=500)
    p_va = model.predict_proba(X_va)[:, 1].astype(np.float32)
    oof_xgb[va_idx] = p_va
    auc = roc_auc_score(y_va, p_va)
    fold_aucs_xgb.append(auc)
    print(f"  [XGB] Fold {fold} AUC: {auc:.6f}")
    pred_xgb += model.predict_proba(X_test_xgb)[:, 1].astype(np.float32) / N_SPLITS

xgb_auc = roc_auc_score(y, oof_xgb)
print(f"\n  [XGB] OOF AUC: {xgb_auc:.6f}  Folds: {[f'{a:.4f}' for a in fold_aucs_xgb]}")
np.save(f"oof_xgb_v{XGB_VER}.npy",  oof_xgb)
np.save(f"pred_xgb_v{XGB_VER}.npy", pred_xgb)


# ============================================================
# ============================================================
# MODEL 3: NEURAL NET (your original EmbMLP)
# ============================================================
# ============================================================
print("\n" + "="*70)
print("MODEL 3: NEURAL NET (EmbMLP)")
print("="*70)

NN_VER = 312

FEATS_NN = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
            'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
            'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
            'MonthlyCharges', 'TotalCharges',
            # engineered features
            'charges_per_tenure', 'total_vs_expected', 'total_minus_expected',
            'n_services', 'is_high_value', 'is_new_customer', 'is_loyal_customer',
            'is_monthly', 'is_autopay']

NUMS_NN    = ["tenure", "MonthlyCharges", "TotalCharges",
              "charges_per_tenure", "total_vs_expected", "total_minus_expected"]
CAT_FEATS_NN = [c for c in FEATS_NN if c not in NUMS_NN]

RARE_MIN_COUNT = 25
NN_EPOCHS      = 100
BATCH_SIZE     = 256
LR_NN          = 2.5e-5
PATIENCE_NN    = 10
WEIGHT_DECAY   = 3e-4
EMB_DROPOUT    = 0.10
MLP_DROPOUT    = 0.30
HIDDEN_NN      = (512, 256)
WARMUP_EPOCHS  = 1
ETA_MIN_NN     = LR_NN * 0.05

def make_vocab_maps(df, cols):
    maps = {}; sizes = {}
    for c in cols:
        uniq = pd.Series(df[c].values).astype(str).unique().tolist()
        v2i = {v: i+1 for i, v in enumerate(uniq)}
        maps[c] = v2i; sizes[c] = len(v2i) + 1
    return maps, sizes

def encode_with_maps(df, cols, maps):
    X = np.zeros((len(df), len(cols)), dtype=np.int64)
    for j, c in enumerate(cols):
        v2i = maps[c]
        X[:, j] = pd.Series(df[c].values).astype(str).map(v2i).fillna(0).astype(np.int64).values
    return X

def emb_dim_from_card(card):
    return int(np.clip(int(round(1.8 * (card ** 0.25))), 4, 64))

def build_numeric_snapper(train_series, rare_min_count):
    s = pd.to_numeric(train_series, errors="coerce").astype(np.float32)
    vc = pd.Series(s).value_counts(dropna=False)
    frequent_vals = vc[vc >= rare_min_count].index.values
    frequent_vals = np.sort(np.array([v for v in frequent_vals if pd.notna(v)], dtype=np.float32))
    if frequent_vals.size == 0:
        frequent_vals = np.sort(np.array(pd.Series(s.dropna()).unique(), dtype=np.float32))
    frequent_set = set(frequent_vals.tolist())

    def transform(series_any):
        x = pd.to_numeric(series_any, errors="coerce").astype(np.float32).values
        is_nan = np.isnan(x)
        is_rare = np.ones_like(x, dtype=np.int32)
        for i, v in enumerate(x):
            if np.isnan(v): is_rare[i] = 1
            else: is_rare[i] = 0 if (float(v) in frequent_set) else 1
        x_snapped = x.copy()
        if frequent_vals.size > 0:
            idx_snap = np.where((~is_nan) & (is_rare == 1))[0]
            if idx_snap.size > 0:
                v = x[idx_snap]
                pos = np.clip(np.searchsorted(frequent_vals, v), 0, len(frequent_vals)-1)
                left = np.clip(pos-1, 0, len(frequent_vals)-1)
                left_vals = frequent_vals[left]; right_vals = frequent_vals[pos]
                nearest = np.where(np.abs(v-right_vals) <= np.abs(v-left_vals), right_vals, left_vals)
                x_snapped[idx_snap] = nearest.astype(np.float32)
        return x_snapped.astype(np.float32), is_rare.astype(np.int32)
    return transform

class TabMixDataset(Dataset):
    def __init__(self, X_cat, X_num, y=None):
        self.Xc = torch.as_tensor(X_cat, dtype=torch.long)
        self.Xn = torch.as_tensor(X_num, dtype=torch.float32)
        self.y  = None if y is None else torch.as_tensor(y, dtype=torch.float32)
    def __len__(self): return self.Xc.shape[0]
    def __getitem__(self, idx):
        if self.y is None: return self.Xc[idx], self.Xn[idx]
        return self.Xc[idx], self.Xn[idx], self.y[idx]

class EmbMLP_Mixed(nn.Module):
    def __init__(self, cat_cardinals, n_num, hidden=(256, 128), emb_dropout=0.1, mlp_dropout=0.2):
        super().__init__()
        self.emb_layers = nn.ModuleList()
        emb_out_dim = 0
        for card in cat_cardinals:
            d = emb_dim_from_card(card)
            self.emb_layers.append(nn.Embedding(num_embeddings=card, embedding_dim=d))
            emb_out_dim += d
        self.emb_drop = nn.Dropout(emb_dropout)
        in_dim = emb_out_dim + n_num
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.LayerNorm(h), nn.ReLU(inplace=True), nn.Dropout(mlp_dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.mlp = nn.Sequential(*layers)
        for emb in self.emb_layers:
            nn.init.normal_(emb.weight, mean=0.0, std=0.02)

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, j]) for j, emb in enumerate(self.emb_layers)]
        z = self.emb_drop(torch.cat(embs, dim=1))
        return self.mlp(torch.cat([z, x_num], dim=1)).squeeze(1)

class SmoothBCE(nn.Module):
    def __init__(self, eps=0.02): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        targets = targets * (1 - self.eps) + 0.5 * self.eps
        return nn.functional.binary_cross_entropy_with_logits(logits, targets)

@torch.no_grad()
def predict_proba_nn(model, loader):
    model.eval()
    out = []
    for batch in loader:
        xc, xn = (batch[0], batch[1])
        logit = model(xc.to(DEVICE), xn.to(DEVICE))
        out.append(torch.sigmoid(logit).cpu().numpy())
    return np.concatenate(out)

def train_one_fold_nn(Xc_tr, Xn_tr, y_tr, Xc_va, Xn_va, y_va, cat_cardinals):
    model = EmbMLP_Mixed(cat_cardinals, Xn_tr.shape[1], HIDDEN_NN, EMB_DROPOUT, MLP_DROPOUT).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_NN, weight_decay=WEIGHT_DECAY)
    loss_fn = SmoothBCE(0.02)

    dl_tr = DataLoader(TabMixDataset(Xc_tr, Xn_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    dl_va = DataLoader(TabMixDataset(Xc_va, Xn_va, y_va), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, NN_EPOCHS - WARMUP_EPOCHS), eta_min=ETA_MIN_NN)

    best_auc, best_state, bad = -1.0, None, 0
    for epoch in range(1, NN_EPOCHS + 1):
        t0 = time.time()
        if epoch <= WARMUP_EPOCHS:
            for pg in opt.param_groups:
                pg["lr"] = LR_NN * (0.1 + 0.9 * (epoch / WARMUP_EPOCHS))

        model.train()
        running_loss = 0.0; n_seen = 0
        for xc, xn, yb in dl_tr:
            xc = xc.to(DEVICE, non_blocking=True)
            xn = xn.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xc, xn), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running_loss += loss.item() * xc.size(0); n_seen += xc.size(0)

        if epoch > WARMUP_EPOCHS: sched.step()
        p_va = predict_proba_nn(model, dl_va)
        auc = roc_auc_score(y_va, p_va)
        print(f"    epoch {epoch:03d} | lr {opt.param_groups[0]['lr']:.2e} | "
              f"loss {running_loss/n_seen:.5f} | val_auc {auc:.6f} | {time.time()-t0:.1f}s")

        if auc > best_auc + 1e-6:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE_NN:
                print(f"    early stop. best_auc={best_auc:.6f}")
                break

    if best_state: model.load_state_dict(best_state)
    return model, best_auc

y_nn = train_fe[TARGET].values
if y_nn.dtype == object:
    y_nn = pd.Series(y_nn).map({"Yes": 1, "No": 0}).values
y_nn = y_nn.astype(np.float32)

skf_nn = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_nn  = np.zeros(N, dtype=np.float32)
pred_nn = np.zeros(len(test_df), dtype=np.float32)
fold_aucs_nn = []

for fold, (tr_idx, va_idx) in enumerate(skf_nn.split(np.zeros(N), y_nn), 1):
    print(f"\n  [NN] Fold {fold}/{N_SPLITS}")
    tr_df_nn = train_fe.iloc[tr_idx].reset_index(drop=True)
    va_df_nn = train_fe.iloc[va_idx].reset_index(drop=True)
    te_df_nn = test_fe

    tr_cat_df = tr_df_nn[CAT_FEATS_NN].copy()
    va_cat_df = va_df_nn[CAT_FEATS_NN].copy()
    te_cat_df = te_df_nn[CAT_FEATS_NN].copy()

    for col in NUMS_NN:
        snapper = build_numeric_snapper(tr_df_nn[col], RARE_MIN_COUNT)
        for df_c, dst in [(tr_df_nn, tr_cat_df), (va_df_nn, va_cat_df), (te_df_nn, te_cat_df)]:
            snap, israre = snapper(df_c[col])
            dst[f"{col}__cat"]    = pd.Series(snap.astype(np.float32)).astype(str).values
            dst[f"{col}__is_rare"]= israre.astype(np.int32)

    CAT_ALL_NN = list(tr_cat_df.columns)
    maps, sizes = make_vocab_maps(tr_cat_df, CAT_ALL_NN)
    cat_cardinals = [sizes[c] for c in CAT_ALL_NN]

    Xc_tr_nn = encode_with_maps(tr_cat_df, CAT_ALL_NN, maps)
    Xc_va_nn = encode_with_maps(va_cat_df, CAT_ALL_NN, maps)
    Xc_te_nn = encode_with_maps(te_cat_df, CAT_ALL_NN, maps)

    scaler_nn = StandardScaler()
    Xn_tr_nn = scaler_nn.fit_transform(tr_df_nn[NUMS_NN].astype(np.float32).values).astype(np.float32)
    Xn_va_nn = scaler_nn.transform(va_df_nn[NUMS_NN].astype(np.float32).values).astype(np.float32)
    Xn_te_nn = scaler_nn.transform(te_df_nn[NUMS_NN].astype(np.float32).values).astype(np.float32)

    y_tr_nn = y_nn[tr_idx]; y_va_nn = y_nn[va_idx]
    model_nn, best_auc_nn = train_one_fold_nn(Xc_tr_nn, Xn_tr_nn, y_tr_nn,
                                               Xc_va_nn, Xn_va_nn, y_va_nn, cat_cardinals)
    fold_aucs_nn.append(best_auc_nn)
    oof_nn[va_idx] = predict_proba_nn(model_nn,
        DataLoader(TabMixDataset(Xc_va_nn, Xn_va_nn, y_va_nn),
                   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True))
    pred_nn += predict_proba_nn(model_nn,
        DataLoader(TabMixDataset(Xc_te_nn, Xn_te_nn, None),
                   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)) / N_SPLITS
    print(f"  [NN] Fold {fold} best AUC: {best_auc_nn:.6f}")

nn_auc = roc_auc_score(y, oof_nn)
print(f"\n  [NN] OOF AUC: {nn_auc:.6f}  Folds: {[f'{a:.4f}' for a in fold_aucs_nn]}")
np.save(f"oof_nn_v{NN_VER}.npy",  oof_nn)
np.save(f"pred_nn_v{NN_VER}.npy", pred_nn)


# ============================================================
# ============================================================
# MODEL 4: LGBM (new)
# ============================================================
# ============================================================
print("\n" + "="*70)
print("MODEL 4: LIGHTGBM")
print("="*70)

lgbm_params = dict(
    objective="binary", metric="auc", learning_rate=0.05,
    num_leaves=31, max_depth=-1, min_child_samples=20,
    subsample=0.80, subsample_freq=1, colsample_bytree=0.75,
    reg_alpha=0.1, reg_lambda=1.0, n_estimators=10_000,
    random_state=SEED, n_jobs=-1, device="gpu", verbose=-1,
)

skf_lgb = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_lgbm  = np.zeros(N, dtype=np.float32)
pred_lgbm = np.zeros(len(test_df), dtype=np.float32)
fold_aucs_lgb = []

for fold, (tr_idx, va_idx) in enumerate(skf_lgb.split(X_train_tree, y), 1):
    X_tr, X_va = X_train_tree.iloc[tr_idx], X_train_tree.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    model_lgb = lgb.LGBMClassifier(**lgbm_params)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(500)])
    p_va = model_lgb.predict_proba(X_va)[:, 1].astype(np.float32)
    oof_lgbm[va_idx] = p_va
    auc = roc_auc_score(y_va, p_va)
    fold_aucs_lgb.append(auc)
    print(f"  [LGBM] Fold {fold} AUC: {auc:.6f} | best iter: {model_lgb.best_iteration_}")
    pred_lgbm += model_lgb.predict_proba(X_test_tree)[:, 1].astype(np.float32) / N_SPLITS

lgbm_auc = roc_auc_score(y, oof_lgbm)
print(f"\n  [LGBM] OOF AUC: {lgbm_auc:.6f}  Folds: {[f'{a:.4f}' for a in fold_aucs_lgb]}")
np.save("oof_lgbm_v1.npy",  oof_lgbm)
np.save("pred_lgbm_v1.npy", pred_lgbm)


# ============================================================
# ============================================================
# MODEL 5: XGB v2 — hand-tuned deeper params (diversity vs Model 2)
# ============================================================
# ============================================================
print("\n" + "="*70)
print("MODEL 5: XGBOOST v2 (hand-tuned deeper params)")
print("="*70)

# Deliberately different from Model 2:
#   - deeper trees (max_depth 5 vs 3)
#   - lower LR with more trees -> smoother fit
#   - stronger colsample + L1 reg for sparsity
#   - lower min_child_weight -> picks up smaller patterns
final_xgb_opt_params = dict(
    n_estimators      = 50_000,
    learning_rate     = 0.02,       # slower + smoother
    max_depth         = 5,          # deeper than Model 2 (was 3)
    min_child_weight  = 2,          # finer splits
    subsample         = 0.80,
    colsample_bytree  = 0.70,
    colsample_bylevel = 0.80,
    gamma             = 0.1,        # min split gain
    reg_alpha         = 0.5,        # L1 sparsity
    reg_lambda        = 2.0,        # L2 smoothing
    objective         = "binary:logistic",
    eval_metric       = "auc",
    tree_method       = "hist",
    enable_categorical= False,
    early_stopping_rounds = 300,
    device            = "cuda",
    random_state      = SEED,
)

skf_xgb2 = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_xgb_opt  = np.zeros(N, dtype=np.float32)
pred_xgb_opt = np.zeros(len(test_df), dtype=np.float32)
fold_aucs_xgb_opt = []

for fold, (tr_idx, va_idx) in enumerate(skf_xgb2.split(X_train_tree, y), 1):
    X_tr, X_va = X_train_tree.iloc[tr_idx], X_train_tree.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    m = xgb.XGBClassifier(**final_xgb_opt_params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=500)
    p_va = m.predict_proba(X_va)[:, 1].astype(np.float32)
    oof_xgb_opt[va_idx] = p_va
    auc = roc_auc_score(y_va, p_va)
    fold_aucs_xgb_opt.append(auc)
    print(f"  [XGB-Opt] Fold {fold} AUC: {auc:.6f}")
    pred_xgb_opt += m.predict_proba(X_test_tree)[:, 1].astype(np.float32) / N_SPLITS

xgb_opt_auc = roc_auc_score(y, oof_xgb_opt)
print(f"\n  [XGB-Opt] OOF AUC: {xgb_opt_auc:.6f}")
np.save("oof_xgb_opt_v1.npy",  oof_xgb_opt)
np.save("pred_xgb_opt_v1.npy", pred_xgb_opt)


# ============================================================
# ============================================================
# ENSEMBLE
# ============================================================
# ============================================================
print("\n" + "="*70)
print("ENSEMBLE")
print("="*70)

# Summary of all models
all_oofs = {
    "tepair":  oof_tepair,
    "xgb":     oof_xgb,
    "nn":      oof_nn,
    "lgbm":    oof_lgbm,
    "xgb_opt": oof_xgb_opt,
}
all_preds = {
    "tepair":  pred_tepair,
    "xgb":     pred_xgb,
    "nn":      pred_nn,
    "lgbm":    pred_lgbm,
    "xgb_opt": pred_xgb_opt,
}

print("\n  Individual model OOF AUCs:")
for name, oof_c in all_oofs.items():
    print(f"    {name:>10s}: {roc_auc_score(y, oof_c):.6f}")

# 1. Rank average
def rank_avg(preds_list):
    ranked = [pd.Series(p).rank(pct=True).values for p in preds_list]
    return np.mean(ranked, axis=0).astype(np.float32)

oof_rank  = rank_avg(list(all_oofs.values()))
pred_rank = rank_avg(list(all_preds.values()))
print(f"\n  Rank-avg OOF AUC:    {roc_auc_score(y, oof_rank):.6f}")

# 2. Simple mean
oof_mean  = np.mean(list(all_oofs.values()), axis=0)
pred_mean = np.mean(list(all_preds.values()), axis=0)
print(f"  Simple-mean OOF AUC: {roc_auc_score(y, oof_mean):.6f}")

# 3. Optimized weights
def optimize_weights(oof_dict, y):
    names = list(oof_dict.keys())
    oofs  = [oof_dict[n] for n in names]
    def neg_auc(w):
        w = np.clip(w, 0, None)
        if w.sum() < 1e-9: return 1.0
        w = w / w.sum()
        return -roc_auc_score(y, sum(wi * oi for wi, oi in zip(w, oofs)))
    res = minimize(neg_auc, np.ones(len(oofs)) / len(oofs), method="Nelder-Mead",
                   options={"maxiter": 5000, "xatol": 1e-6, "fatol": 1e-6})
    w = np.clip(res.x, 0, None); w /= w.sum()
    return dict(zip(names, w.tolist()))

best_weights = optimize_weights(all_oofs, y)
print(f"\n  Optimized weights:")
for name, w in best_weights.items():
    print(f"    {name:>10s}: {w:.4f}")
oof_opt  = np.array(sum(best_weights[n] * all_oofs[n]  for n in best_weights), dtype=np.float32)
pred_opt = np.array(sum(best_weights[n] * all_preds[n] for n in best_weights), dtype=np.float32)
print(f"  Optimized blend OOF AUC: {roc_auc_score(y, oof_opt):.6f}")

# 4. Level-2 stacking (meta-logit)
skf_stack = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
model_names_stack = list(all_oofs.keys())
oof_mat  = np.column_stack([all_oofs[n]  for n in model_names_stack])
test_mat = np.column_stack([all_preds[n] for n in model_names_stack])

oof_stack  = np.zeros(N, dtype=np.float32)
pred_stack = np.zeros(len(test_df), dtype=np.float32)

def to_logit_arr(x, eps=1e-5):
    x = np.clip(x, eps, 1-eps)
    return np.log(x / (1-x)).astype(np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf_stack.split(oof_mat, y), 1):
    scaler_s = StandardScaler()
    Xl_tr = scaler_s.fit_transform(to_logit_arr(oof_mat[tr_idx]))
    Xl_va = scaler_s.transform(to_logit_arr(oof_mat[va_idx]))
    Xl_te = scaler_s.transform(to_logit_arr(test_mat))
    meta_s = LogisticRegression(C=0.5, max_iter=2000, random_state=SEED)
    meta_s.fit(Xl_tr, y[tr_idx])
    oof_stack[va_idx] = meta_s.predict_proba(Xl_va)[:, 1].astype(np.float32)
    pred_stack += meta_s.predict_proba(Xl_te)[:, 1].astype(np.float32) / N_SPLITS

print(f"  Stacking OOF AUC:    {roc_auc_score(y, oof_stack):.6f}")

# Pick best
candidates = {
    "rank_avg":  (oof_rank,  pred_rank),
    "mean":      (oof_mean,  pred_mean),
    "opt_blend": (oof_opt,   pred_opt),
    "stack":     (oof_stack, pred_stack),
}

print("\n  Final comparison:")
best_name, best_auc_final, best_pred_final = None, -1, None
for name, (oof_c, pred_c) in candidates.items():
    auc = roc_auc_score(y, oof_c)
    tag = "  <-- BEST" if auc > best_auc_final else ""
    print(f"    {name:>12s}: {auc:.6f}{tag}")
    if auc > best_auc_final:
        best_auc_final = auc; best_name = name; best_pred_final = pred_c

print(f"\n  Winner: {best_name}  OOF AUC: {best_auc_final:.6f}")

sub = test_df[["id"]].copy()
sub[TARGET] = best_pred_final
sub.to_csv("submission_final.csv", index=False)

np.save("oof_final.npy",  candidates[best_name][0])
np.save("pred_final.npy", best_pred_final)

print("\nDone. Saved:")
print("  submission_final.csv")
print("  oof_final.npy  /  pred_final.npy")
print(f"  All individual oof_*.npy and pred_*.npy files")

# ---- Save a submission CSV for EVERY individual model ----
print("\n  Saving individual model submissions...")
for name, pred in all_preds.items():
    auc = roc_auc_score(y, all_oofs[name])
    fname = f"submission_{name}.csv"
    sub = test_df[["id"]].copy()
    sub[TARGET] = pred.astype(np.float32)
    sub.to_csv(fname, index=False)
    print(f"    {fname:<35s}  OOF AUC={auc:.6f}")

# ---- Save a submission CSV for EVERY ensemble strategy ----
print("\n  Saving ensemble submissions...")
best_name, best_auc_final, best_pred_final = None, -1, None
for name, (oof_c, pred_c) in candidates.items():
    auc = roc_auc_score(y, oof_c)
    fname = f"submission_ensemble_{name}.csv"
    sub = test_df[["id"]].copy()
    sub[TARGET] = pred_c.astype(np.float32)
    sub.to_csv(fname, index=False)
    tag = ""
    if auc > best_auc_final:
        best_auc_final = auc; best_name = name; best_pred_final = pred_c
        tag = "  <-- BEST"
    print(f"    {fname:<45s}  OOF AUC={auc:.6f}{tag}")

# ---- Also copy the winner as submission_final.csv for convenience ----
sub = test_df[["id"]].copy()
sub[TARGET] = best_pred_final.astype(np.float32)
sub.to_csv("submission_final.csv", index=False)

print(f"\n  Winner: {best_name}  OOF AUC: {best_auc_final:.6f}")
print(f"  Also saved as: submission_final.csv")

print("\nDone. All files saved:")
print("  Individual models:")
for name in all_preds:
    print(f"    submission_{name}.csv")
print("  Ensemble strategies:")
for name in candidates:
    print(f"    submission_ensemble_{name}.csv")
print("  submission_final.csv  (best ensemble copy)")

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Loading data...
  train: (594194, 21)  test: (254655, 20)
Tree features: 45

MODEL 1: TE-PAIR LOGISTIC REGRESSION

  [TE-Pair] Outer fold 1/5  tr=475355 va=118839
    Pair      1/171: gender x SeniorCitizen
    Pair    171/171: MonthlyCharges x TotalCharges
    [TE-Pair] Fold 1 AUC: 0.915793

  [TE-Pair] Outer fold 2/5  tr=475355 va=118839
    Pair      1/171: gender x SeniorCitizen
    Pair    171/171: MonthlyCharges x TotalCharges
    [TE-Pair] Fold 2 AUC: 0.916724

  [TE-Pair] Outer fold 3/5  tr=475355 va=118839
    Pair      1/171: gender x SeniorCitizen
    Pair    171/171: MonthlyCharges x TotalCharges
    [TE-Pair] Fold 3 AUC: 0.916048

  [TE-Pair] Outer fold 4/5  tr=475355 va=118839
    Pair      1/171: gender x SeniorCitizen
    Pair    171/171: MonthlyCharges x TotalCharges
    [TE-Pair] Fold 4 AUC: 0.917332

  [TE-Pair] Outer fold 5/5  tr=475356 va=118838
    Pair      1/171: gender x SeniorCitizen
    Pair    171/171: MonthlyCharges x TotalCharges
    [TE-Pair] Fold 5 AUC: 

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [22:22:54] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  [XGB] Fold 1 AUC: 0.916277
[0]	validation_0-auc:0.88223
[500]	validation_0-auc:0.91674
[1000]	validation_0-auc:0.91711
[1500]	validation_0-auc:0.91717
[1656]	validation_0-auc:0.91716
  [XGB] Fold 2 AUC: 0.917191
[0]	validation_0-auc:0.88162
[500]	validation_0-auc:0.91622
[1000]	validation_0-auc:0.91663
[1500]	validation_0-auc:0.91671
[1868]	validation_0-auc:0.91671
  [XGB] Fold 3 AUC: 0.916739
[0]	validation_0-auc:0.88295
[500]	validation_0-auc:0.91735
[1000]	validation_0-auc:0.91782
[1500]	validation_0-auc:0.91790
[1978]	validation_0-auc:0.91791
  [XGB] Fold 4 AUC: 0.917931
[0]	validation_0-auc:0.88027
[500]	validation_0-auc:0.91451
[1000]	validation_0-auc:0.91492
[1500]	validation_0-auc:0.91492
[1543]	validation_0-auc:0.91493
  [XGB] Fold 5 AUC: 0.914961

  [XGB] OOF AUC: 0.916613  Folds: ['0.9163', '0.9172', '0.9167', '0.9179', '0.9150']

MODEL 3: NEURAL NET (EmbMLP)

  [NN] Fold 1/5
    epoch 001 | lr 2.50e-05 | loss 0.35792 | val_auc 0.908954 | 18.5s
    epoch 002 | lr 2.50e-05 

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[500]	valid_0's auc: 0.915876
[1000]	valid_0's auc: 0.915947
  [LGBM] Fold 1 AUC: 0.915961 | best iter: 901
[500]	valid_0's auc: 0.916871
[1000]	valid_0's auc: 0.916987
  [LGBM] Fold 2 AUC: 0.916992 | best iter: 1024
[500]	valid_0's auc: 0.916308
  [LGBM] Fold 3 AUC: 0.916436 | best iter: 726
[500]	valid_0's auc: 0.917348
[1000]	valid_0's auc: 0.917444
  [LGBM] Fold 4 AUC: 0.917455 | best iter: 931
[500]	valid_0's auc: 0.91464
[1000]	valid_0's auc: 0.914627
  [LGBM] Fold 5 AUC: 0.914696 | best iter: 869

  [LGBM] OOF AUC: 0.916300  Folds: ['0.9160', '0.9170', '0.9164', '0.9175', '0.9147']

MODEL 5: XGBOOST v2 (hand-tuned deeper params)
[0]	validation_0-auc:0.89527
[500]	validation_0-auc:0.91468
[1000]	validation_0-auc:0.91575
[1500]	validation_0-auc:0.91609
[2000]	validation_0-auc:0.91623
[2500]	validation_0-auc:0.91624
[2712]	validation_0-auc:0.91624
  [XGB-Opt] Fold 1 AUC: 0.916245
[0]	validation_0-auc:0.89608
[500]	validation_0-auc:0.91584
[1000]	validation_0-auc:0.91680
[1500]	vali